In [4]:
import os
import json

In [8]:
os.getcwd()

'C:\\Users\\bonse\\Documents\\Work\\Data Science\\Nebius AI Performance Engineer\\Module 3 - MLOps\\assignment - 02\\evals'

In [10]:
data = []



In [11]:
with open("eval_set.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(data)


[{'question': 'What is the coordinates location of the circuits for Australian grand prix?', 'db_id': 'formula_1', 'gold_sql': "SELECT DISTINCT T1.lat, T1.lng FROM circuits AS T1 INNER JOIN races AS T2 ON T2.circuitID = T1.circuitId WHERE T2.name = 'Australian Grand Prix'"}, {'question': "List down Ajax's superpowers.", 'db_id': 'superhero', 'gold_sql': "SELECT T3.power_name FROM superhero AS T1 INNER JOIN hero_power AS T2 ON T1.id = T2.hero_id INNER JOIN superpower AS T3 ON T2.power_id = T3.id WHERE T1.superhero_name = 'Ajax'"}, {'question': 'List the top five schools, by descending order, from the highest to the lowest, the most number of Enrollment (Ages 5-17). Please give their NCES school identification number.', 'db_id': 'california_schools', 'gold_sql': 'SELECT T1.NCESSchool FROM schools AS T1 INNER JOIN frpm AS T2 ON T1.CDSCode = T2.CDSCode ORDER BY T2.`Enrollment (Ages 5-17)` DESC LIMIT 5'}, {'question': 'What is the average number of crimes committed in 1995 in regions where 

In [19]:
data[0].keys()

dict_keys(['question', 'db_id', 'gold_sql'])

In [22]:
for line in data:
    print(line['question'])
    print(line['gold_sql'])

What is the coordinates location of the circuits for Australian grand prix?
SELECT DISTINCT T1.lat, T1.lng FROM circuits AS T1 INNER JOIN races AS T2 ON T2.circuitID = T1.circuitId WHERE T2.name = 'Australian Grand Prix'
List down Ajax's superpowers.
SELECT T3.power_name FROM superhero AS T1 INNER JOIN hero_power AS T2 ON T1.id = T2.hero_id INNER JOIN superpower AS T3 ON T2.power_id = T3.id WHERE T1.superhero_name = 'Ajax'
List the top five schools, by descending order, from the highest to the lowest, the most number of Enrollment (Ages 5-17). Please give their NCES school identification number.
SELECT T1.NCESSchool FROM schools AS T1 INNER JOIN frpm AS T2 ON T1.CDSCode = T2.CDSCode ORDER BY T2.`Enrollment (Ages 5-17)` DESC LIMIT 5
What is the average number of crimes committed in 1995 in regions where the number exceeds 4000 and the region has accounts that are opened starting from the year 1997?
SELECT AVG(T1.A15) FROM district AS T1 INNER JOIN account AS T2 ON T1.district_id = T2.di

In [1]:
import json
import sqlite3
import os

In [9]:

EVAL_PATH = 'eval_set.jsonl'  # adjust if needed
DB_ROOT = './../data/bird/'        # base directory containing BIRD .db files


In [10]:

def load_eval(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

In [15]:


def find_db_path(db_root, db_id):
    # common patterns: db_id.db or db_id/database.sqlite
    candidates = [
        os.path.join(db_root, f"{db_id}.sqlite"),
        os.path.join(db_root, db_id, f"{db_id}.db"),
        os.path.join(db_root, db_id, 'database.sqlite'),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

In [23]:
def run_eval(eval_items, db_root):
    results = []
    for idx, item in enumerate(eval_items):
        q = item.get('question')
        db_id = item.get('db_id')
        sql = item.get('gold_sql')
        rec = {
            'index': idx,
            'question': q,
            'db_id': db_id,
            'sql': sql,
            'status': 'not_run',
            'error': None,
            'rowcount': None,
            'sample_row': None,
        }
        db_path = find_db_path(db_root, db_id)
        if not db_path:
            rec['status'] = 'db_not_found'
            results.append(rec)
            continue
        try:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()
            cur.execute(sql)
            rows = cur.fetchall()
            rec['status'] = 'ok'
            rec['rowcount'] = len(rows)
            rec['rows'] = rows
        except Exception as e:
            rec['status'] = 'error'
            rec['error'] = str(e)
        finally:
            try:
                conn.close()
            except Exception:
                pass
        results.append(rec)
    return results

In [24]:
eval_items = load_eval(EVAL_PATH)
results = run_eval(eval_items, DB_ROOT)




In [25]:
results

[{'index': 0,
  'question': 'What is the coordinates location of the circuits for Australian grand prix?',
  'db_id': 'formula_1',
  'sql': "SELECT DISTINCT T1.lat, T1.lng FROM circuits AS T1 INNER JOIN races AS T2 ON T2.circuitID = T1.circuitId WHERE T2.name = 'Australian Grand Prix'",
  'status': 'ok',
  'error': None,
  'rowcount': 1,
  'sample_row': None,
  'rows': [(-34.9272, 138.617)]},
 {'index': 1,
  'question': "List down Ajax's superpowers.",
  'db_id': 'superhero',
  'sql': "SELECT T3.power_name FROM superhero AS T1 INNER JOIN hero_power AS T2 ON T1.id = T2.hero_id INNER JOIN superpower AS T3 ON T2.power_id = T3.id WHERE T1.superhero_name = 'Ajax'",
  'status': 'ok',
  'error': None,
  'rowcount': 5,
  'sample_row': None,
  'rows': [('Agility',),
   ('Super Strength',),
   ('Super Speed',),
   ('Heat Generation',),
   ('Power Suit',)]},
 {'index': 2,
  'question': 'List the top five schools, by descending order, from the highest to the lowest, the most number of Enrollment (

In [27]:
for id,result in enumerate(results):
    print(id,': ', result['question'])
    print(result['rows'])
    print('='*25)

0 :  What is the coordinates location of the circuits for Australian grand prix?
[(-34.9272, 138.617)]
1 :  List down Ajax's superpowers.
[('Agility',), ('Super Strength',), ('Super Speed',), ('Heat Generation',), ('Power Suit',)]
2 :  List the top five schools, by descending order, from the highest to the lowest, the most number of Enrollment (Ages 5-17). Please give their NCES school identification number.
[('11707',), ('04653',), ('08283',), ('02751',), ('03050',)]
3 :  What is the average number of crimes committed in 1995 in regions where the number exceeds 4000 and the region has accounts that are opened starting from the year 1997?
[(29670.44951923077,)]
4 :  How many male clients in 'Hl.m. Praha' district?
[(339,)]
5 :  What is the average fastest lap time in seconds for Lewis Hamilton in all the Formula_1 races?
[(92.01671065989848,)]
6 :  From race no. 50 to 100, how many finishers have been disqualified?
[(2,)]
7 :  Calculate the difference of the total amount spent in all e